# Stage 2 Feature Engineering

这版 notebook 只做 `Stage 2` 特征工程，不进入建模。

目标：
- 读取 `Stage 1` 的 selected `OOF / hold-out` 概率
- 保留运行时指定的 `P_taco` 版本
- 将 event-level 概率聚合成 daily TACO 特征
- 与 lagged market / macro / Google Trends 控制变量合并
- 产出后续 `Stage 2` 建模可直接使用的 daily 表

当前 notebook 不做：
- 模型比较
- 调参
- SHAP
- with vs without FinBERT / raw-vs-calibrated 的 Stage 2 对照

## Output Design

这版会导出 3 份主要结果：
- `event_bridge`: event-level 到 daily 的桥接表，方便审计
- `daily_long_v2`: 每天每个 `P_taco` 版本各一行，加入新的 aggregation v2 特征
- `daily_wide_v2`: 每天一行、不同 `P_taco` 版本做宽表前缀展开，适合快速对比

说明：
- `Stage 2` 主样本仍以 trading-day 为单位
- `gold_ret_1d` 作为当前 daily 目标列保留在表中
- 控制变量统一先做 `lag1`，避免使用目标日收盘后才知道的信息
- `aggregation v2` 会在保留原始 `max/sum/mean` 的同时，补充更贴近交易逻辑的强信号、阈值 tail、noisy-or 与跨日衰减特征

# Dense Export Sensitivity Stage 2 Feature Runtime Config

这个 notebook 固定用于 `dense export sensitivity` 的 Stage 2 feature engineering：

- `SPLIT_SCHEME = "primary_holdout"`
- `Stage 1 winner` 冻结继承主链：`linear_svm__sigmoid`
- 输入固定读取 dense export sensitivity 的 `Stage 1 OOF / holdout` 预测
- 输出目录固定到 `main_rerun/artifacts/dense_export_sensitivity/stage2/`
- 不覆盖 `main_rerun/artifacts/stage2/`

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

DENSE_STAGE1_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "dense_export_sensitivity" / "stage1"
DENSE_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "dense_export_sensitivity" / "stage2"
DENSE_STAGE2_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE_SPLIT_SCHEME"] = "primary_holdout"
os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE1_RESULTS_DIR"] = str(DENSE_STAGE1_DIR)
os.environ["STAGE1_OOF_PATH"] = str(DENSE_STAGE1_DIR / "stage1_v3_oof_predictions_selected.csv")
os.environ["STAGE1_HOLDOUT_PATH"] = str(
    DENSE_STAGE1_DIR / "stage1_v3_primary_holdout_predictions_selected.csv"
)
os.environ["STAGE2_FEATURE_RESULTS_DIR"] = str(DENSE_STAGE2_DIR)
os.environ["STAGE2_ACTIVE_VARIANTS"] = "linear_svm__sigmoid"
os.environ["STAGE2_ACTIVE_VARIANT_SPECS_JSON"] = '{"linear_svm__sigmoid": {"model_name": "linear_svm", "prob_version": "sigmoid"}}'
os.environ["STAGE2_EVENT_BRIDGE_PATH"] = str(
    DENSE_STAGE2_DIR / "stage2_event_daily_bridge_dense_export_sensitivity.csv"
)
os.environ["STAGE2_DAILY_LONG_PATH"] = str(
    DENSE_STAGE2_DIR / "stage2_daily_features_long_v2_dense_export_sensitivity.csv"
)
os.environ["STAGE2_DAILY_WIDE_PATH"] = str(
    DENSE_STAGE2_DIR / "stage2_daily_features_wide_v2_dense_export_sensitivity.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dense export Stage 1 input dir:", DENSE_STAGE1_DIR)
print("Dense export Stage 2 output dir:", DENSE_STAGE2_DIR)
print("Frozen Stage 1 variant:", "linear_svm__sigmoid")

In [ ]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

ROOT = Path.cwd()
STAGE1_DIR = Path(os.environ.get("STAGE1_RESULTS_DIR", ROOT / "stage1" / "artifacts" / "v3"))
INTEGRATION_DIR = ROOT / "data" / "integration"
RESULTS_DIR = Path(os.environ.get("STAGE2_FEATURE_RESULTS_DIR", ROOT / "stage2" / "artifacts"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OOF_PATH = Path(os.environ.get("STAGE1_OOF_PATH", STAGE1_DIR / "stage1_v3_oof_predictions_selected.csv"))
HOLDOUT_PATH = Path(
    os.environ.get(
        "STAGE1_HOLDOUT_PATH",
        STAGE1_DIR / "stage1_v3_primary_holdout_predictions_selected.csv",
    )
)
MARKET_PATH = INTEGRATION_DIR / "market_macro_features_daily.csv"
TRENDS_PATH = INTEGRATION_DIR / "trends_features_daily.csv"
BRIDGE_PATH = Path(
    os.environ.get(
        "STAGE2_EVENT_BRIDGE_PATH",
        str(RESULTS_DIR / "stage2_event_daily_bridge_v1.csv"),
    )
)
LONG_PATH = Path(
    os.environ.get(
        "STAGE2_DAILY_LONG_PATH",
        str(RESULTS_DIR / "stage2_daily_features_long_v2.csv"),
    )
)
WIDE_PATH = Path(
    os.environ.get(
        "STAGE2_DAILY_WIDE_PATH",
        str(RESULTS_DIR / "stage2_daily_features_wide_v2.csv"),
    )
)

VARIANT_SPECS = {
    "svm_sigmoid": {
        "model_name": "linear_svm",
        "prob_version": "sigmoid",
    },
    "xgb_sigmoid": {
        "model_name": "xgboost_classifier",
        "prob_version": "sigmoid",
    },
}
runtime_variant_specs_text = os.environ.get("STAGE2_ACTIVE_VARIANT_SPECS_JSON", "").strip()
if runtime_variant_specs_text:
    runtime_variant_specs = json.loads(runtime_variant_specs_text)
    if not isinstance(runtime_variant_specs, dict) or not runtime_variant_specs:
        raise ValueError("STAGE2_ACTIVE_VARIANT_SPECS_JSON must be a non-empty JSON object.")
    VARIANT_SPECS = runtime_variant_specs

active_variant_default = ",".join(VARIANT_SPECS.keys())
ACTIVE_VARIANT_KEYS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_VARIANTS", active_variant_default).split(",")
    if key.strip()
]
unknown_variant_keys = [key for key in ACTIVE_VARIANT_KEYS if key not in VARIANT_SPECS]
if unknown_variant_keys:
    raise ValueError(f"Unknown STAGE2_ACTIVE_VARIANTS entries: {unknown_variant_keys}")
ACTIVE_VARIANTS = {key: VARIANT_SPECS[key] for key in ACTIVE_VARIANT_KEYS}

HIGH_P_TACO_THRESHOLD = 0.10
TAIL_THRESHOLD_005 = 0.05
TAIL_THRESHOLD_010 = 0.10

In [ ]:
SPLIT_SCHEME = os.environ.get("STAGE_SPLIT_SCHEME", "primary_holdout")

TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
OUTSIDE_LABEL = "outside_main_sample"

SPLIT_SCHEMES = {
    "primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-14"),
        ],
        "holdout_range": ("2025-07-15", "2025-10-25"),
    },
    "first_term_final_year_holdout": {
        "train_ranges": [
            ("2017-01-20", "2019-06-30"),
        ],
        "holdout_range": ("2019-07-01", "2020-02-28"),
    },
}


def describe_split_scheme(split_scheme):
    scheme = SPLIT_SCHEMES[split_scheme]
    return {
        "split_scheme": split_scheme,
        "train_ranges": scheme["train_ranges"],
        "holdout_range": scheme["holdout_range"],
        "train_label": TRAIN_LABEL,
        "holdout_label": HOLDOUT_LABEL,
    }


def classify_daily_split(date_values, split_scheme=SPLIT_SCHEME):
    scheme = SPLIT_SCHEMES[split_scheme]
    dates = pd.to_datetime(pd.Series(date_values)).dt.normalize()
    tz = getattr(dates.dt, "tz", None)
    if tz is not None:
        dates = dates.dt.tz_convert(None)

    train_mask = pd.Series(False, index=dates.index)
    for start_text, end_text in scheme["train_ranges"]:
        start = pd.Timestamp(start_text)
        end = pd.Timestamp(end_text)
        train_mask |= (dates >= start) & (dates <= end)

    holdout_start, holdout_end = scheme["holdout_range"]
    holdout_mask = (dates >= pd.Timestamp(holdout_start)) & (dates <= pd.Timestamp(holdout_end))
    return np.select(
        [train_mask, holdout_mask],
        [TRAIN_LABEL, HOLDOUT_LABEL],
        default=OUTSIDE_LABEL,
    )


print("Split scheme:", describe_split_scheme(SPLIT_SCHEME))

## Aggregation V2 Design

在保留 `v1` 的 `max / sum / mean / event_count` 之外，这版新增 4 类 `P_taco` 日聚合：
- 强信号摘要：`p_taco_top1`、`p_taco_top2_sum`
- “至少有一个强信号”的 noisy-or：`p_taco_any = 1 - Π(1 - p_i)`
- 阈值 tail：`p_taco_tail_005`、`p_taco_tail_010`
- 跨日衰减与 stress interaction：如 `p_taco_any_decay_2d`、`p_taco_any_x_vix_ma_5d_lag1`

目标不是机械累加所有小概率事件，而是把 event-level 概率变成更适合 `Stage 2` 的 daily trading signal。

In [ ]:
def classify_stage2_date(date_series):
    return classify_daily_split(
        date_series,
        split_scheme=SPLIT_SCHEME,
    )


def load_stage1_predictions():
    oof = pd.read_csv(OOF_PATH)
    holdout = pd.read_csv(HOLDOUT_PATH)

    oof["prediction_source"] = "oof"
    holdout["prediction_source"] = "holdout"
    if "fold_id" not in holdout.columns:
        holdout["fold_id"] = np.nan

    stage1 = pd.concat([oof, holdout], ignore_index=True, sort=False)
    stage1["event_time_utc"] = pd.to_datetime(stage1["event_time_utc"], format="mixed", utc=True)
    stage1["feature_anchor_date"] = pd.to_datetime(stage1["feature_anchor_date"], format="mixed", utc=True)
    stage1["target_trade_date"] = pd.to_datetime(stage1["target_trade_date"], format="mixed", utc=True)
    stage1["target_trade_date_naive"] = stage1["target_trade_date"].dt.tz_convert(None).dt.normalize()

    keep_frames = []
    for variant_key, spec in ACTIVE_VARIANTS.items():
        subset = stage1[
            (stage1["model_name"] == spec["model_name"])
            & (stage1["prob_version"] == spec["prob_version"])
        ].copy()
        subset["variant_key"] = variant_key
        keep_frames.append(subset)

    if not keep_frames:
        raise ValueError("No Stage 1 predictions matched the active Stage 2 variant specs.")
    selected = pd.concat(keep_frames, ignore_index=True)
    return selected


def aggregate_event_probabilities(events):
    grouped_rows = []
    group_keys = ["variant_key", "target_trade_date_naive"]
    for (variant_key, target_date), group in events.groupby(group_keys):
        prob = group["p_retreat"].astype(float).clip(0.0, 1.0)
        grouped_rows.append(
            {
                "variant_key": variant_key,
                "date": target_date,
                "max_p_taco": float(prob.max()),
                "sum_p_taco": float(prob.sum()),
                "mean_p_taco": float(prob.mean()),
                "event_count": int(len(group)),
                "high_p_taco_count": int((prob >= HIGH_P_TACO_THRESHOLD).sum()),
                "p_taco_top1": float(prob.max()),
                "p_taco_top2_sum": float(prob.nlargest(min(2, len(prob))).sum()),
                "p_taco_any": float(1.0 - np.prod(1.0 - prob.to_numpy())),
                "p_taco_tail_005": float(np.maximum(prob - TAIL_THRESHOLD_005, 0.0).sum()),
                "p_taco_tail_010": float(np.maximum(prob - TAIL_THRESHOLD_010, 0.0).sum()),
                "event_count_ge_005": int((prob >= TAIL_THRESHOLD_005).sum()),
                "event_count_ge_010": int((prob >= TAIL_THRESHOLD_010).sum()),
                "event_count_ge_020": int((prob >= 0.20).sum()),
                "source_positive_count": int(group["label_retreat"].sum()),
                "min_event_time_utc": group["event_time_utc"].min(),
                "max_event_time_utc": group["event_time_utc"].max(),
                "prediction_source_first": group["prediction_source"].iloc[0],
            }
        )
    grouped = pd.DataFrame(grouped_rows).sort_values(["variant_key", "date"]).reset_index(drop=True)
    return grouped


def build_daily_base():
    market = pd.read_csv(MARKET_PATH)
    trends = pd.read_csv(TRENDS_PATH)

    market["date"] = pd.to_datetime(market["date"]).dt.normalize()
    trends["date"] = pd.to_datetime(trends["date"]).dt.normalize()

    daily = market.merge(trends, on="date", how="left", validate="one_to_one")
    daily = daily.sort_values("date").reset_index(drop=True)
    daily["stage2_split"] = classify_stage2_date(daily["date"])
    daily["is_main_stage2_sample"] = daily["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL]).astype(int)
    return daily


def add_lagged_controls(daily):
    lag_cols = [
        "dxy_ret_1d",
        "real_yield_change_1d",
        "real_yield_change_5d",
        "vix_ma_5d",
        "vix_change_5d",
        "spx_ret_1d",
        "spx_drawdown_5d",
        "gold_spx_corr_20d",
        "trend_tariff_z_60d",
        "trend_inflation_z_60d",
        "trend_war_z_60d",
        "trend_tariff_spike",
        "trend_inflation_spike",
        "trend_war_spike",
    ]
    out = daily.copy()
    for col in lag_cols:
        out[f"{col}_lag1"] = out[col].shift(1)
    return out


def build_stage2_long_table(daily_base, aggregated_events):
    long_frames = []
    for variant_key in ACTIVE_VARIANTS:
        taco = aggregated_events[aggregated_events["variant_key"] == variant_key].copy()
        merged = daily_base.merge(
            taco.drop(columns=["variant_key"]),
            on="date",
            how="left",
            validate="one_to_one",
        )
        merged["variant_key"] = variant_key
        merged["max_p_taco"] = merged["max_p_taco"].fillna(0.0)
        merged["sum_p_taco"] = merged["sum_p_taco"].fillna(0.0)
        merged["mean_p_taco"] = merged["mean_p_taco"].fillna(0.0)
        merged["event_count"] = merged["event_count"].fillna(0).astype(int)
        merged["high_p_taco_count"] = merged["high_p_taco_count"].fillna(0).astype(int)
        merged["p_taco_top1"] = merged["p_taco_top1"].fillna(0.0)
        merged["p_taco_top2_sum"] = merged["p_taco_top2_sum"].fillna(0.0)
        merged["p_taco_any"] = merged["p_taco_any"].fillna(0.0)
        merged["p_taco_tail_005"] = merged["p_taco_tail_005"].fillna(0.0)
        merged["p_taco_tail_010"] = merged["p_taco_tail_010"].fillna(0.0)
        merged["event_count_ge_005"] = merged["event_count_ge_005"].fillna(0).astype(int)
        merged["event_count_ge_010"] = merged["event_count_ge_010"].fillna(0).astype(int)
        merged["event_count_ge_020"] = merged["event_count_ge_020"].fillna(0).astype(int)
        merged["source_positive_count"] = merged["source_positive_count"].fillna(0).astype(int)
        merged["has_stage1_signal"] = (merged["event_count"] > 0).astype(int)
        merged["holdout_spillover_flag"] = (
            (merged["prediction_source_first"] == "holdout")
            & (merged["stage2_split"] == "outside_main_sample")
        ).astype(int)
        long_frames.append(merged)
    long_df = pd.concat(long_frames, ignore_index=True)
    return long_df


def add_temporal_taco_features(long_df):
    signal_cols = [
        "p_taco_any",
        "p_taco_top1",
        "p_taco_top2_sum",
        "p_taco_tail_005",
        "p_taco_tail_010",
    ]
    out_frames = []
    for variant_key, frame in long_df.groupby("variant_key", sort=False):
        ordered = frame.sort_values("date").copy()
        for col in signal_cols:
            ordered[f"{col}_decay_2d"] = (
                ordered[col]
                + 0.5 * ordered[col].shift(1).fillna(0.0)
            )
            ordered[f"{col}_decay_3d"] = (
                ordered[col]
                + 0.5 * ordered[col].shift(1).fillna(0.0)
                + 0.25 * ordered[col].shift(2).fillna(0.0)
            )

        ordered["p_taco_any_x_vix_ma_5d_lag1"] = ordered["p_taco_any"] * ordered["vix_ma_5d_lag1"].fillna(0.0)
        ordered["p_taco_any_x_spx_drawdown_5d_lag1"] = (
            ordered["p_taco_any"] * ordered["spx_drawdown_5d_lag1"].fillna(0.0)
        )
        ordered["p_taco_any_x_dxy_ret_1d_lag1"] = (
            ordered["p_taco_any"] * ordered["dxy_ret_1d_lag1"].fillna(0.0)
        )
        out_frames.append(ordered)
    return pd.concat(out_frames, ignore_index=True)


def build_stage2_wide_table(long_df):
    id_cols = [
        "date",
        "stage2_split",
        "is_main_stage2_sample",
        "gold_close",
        "gold_ret_1d",
        "spx_close",
        "vix_close",
        "dxy_close",
        "real_yield_10y",
    ]
    lag_cols = [col for col in long_df.columns if col.endswith("_lag1")]
    keep_taco_cols = [
        col
        for col in long_df.columns
        if (
            col.startswith("p_taco_")
            or col.startswith("event_count")
            or col in ["max_p_taco", "sum_p_taco", "mean_p_taco", "high_p_taco_count", "has_stage1_signal", "holdout_spillover_flag"]
        )
    ]

    wide_base = (
        long_df[id_cols + lag_cols]
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    taco_wide = (
        long_df[["date", "variant_key"] + keep_taco_cols]
        .set_index(["date", "variant_key"])
        .unstack("variant_key")
    )
    taco_wide.columns = [
        f"{variant}__{feature}"
        for feature, variant in taco_wide.columns.to_flat_index()
    ]
    taco_wide = taco_wide.reset_index()

    wide = wide_base.merge(taco_wide, on="date", how="left", validate="one_to_one")
    return wide

In [ ]:
stage1_events = load_stage1_predictions()
aggregated_events = aggregate_event_probabilities(stage1_events)
daily_base = build_daily_base()
daily_base_lagged = add_lagged_controls(daily_base)
stage2_long = build_stage2_long_table(daily_base_lagged, aggregated_events)
stage2_long = add_temporal_taco_features(stage2_long)
stage2_wide = build_stage2_wide_table(stage2_long)

print("stage1 selected event rows:", stage1_events.shape)
print("aggregated event rows:", aggregated_events.shape)
print("daily base rows:", daily_base.shape)
print("stage2_long rows:", stage2_long.shape)
print("stage2_wide rows:", stage2_wide.shape)

In [ ]:
event_summary = (
    stage1_events.groupby(["variant_key", "prediction_source"])["label_retreat"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "n_events", "sum": "n_positive", "mean": "positive_rate"})
)
display(event_summary)

aggregated_summary = (
    aggregated_events.groupby("variant_key")[
        [
            "max_p_taco",
            "sum_p_taco",
            "mean_p_taco",
            "event_count",
            "high_p_taco_count",
            "p_taco_any",
            "p_taco_top2_sum",
            "p_taco_tail_010",
            "event_count_ge_010",
        ]
    ]
    .agg(["mean", "max"])
)
display(aggregated_summary)

split_summary = (
    stage2_long.groupby(["variant_key", "stage2_split"])
    .agg(
        n_rows=("date", "count"),
        n_signal_days=("has_stage1_signal", "sum"),
        avg_event_count=("event_count", "mean"),
        avg_max_p_taco=("max_p_taco", "mean"),
    )
)
display(split_summary)

In [ ]:
spillover_rows = stage2_long[stage2_long["holdout_spillover_flag"] == 1][
    ["date", "variant_key", "event_count", "max_p_taco", "prediction_source_first", "stage2_split"]
].drop_duplicates()
print("spillover rows outside main sample:", len(spillover_rows))
if len(spillover_rows):
    display(spillover_rows)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

plot_long = stage2_long[stage2_long["is_main_stage2_sample"] == 1].copy()
sns.lineplot(
    data=plot_long,
    x="date",
    y="max_p_taco",
    hue="variant_key",
    ax=axes[0],
    linewidth=1.8,
)
axes[0].set_title("Daily max_p_taco by variant")
axes[0].set_xlabel("")
axes[0].set_ylabel("max_p_taco")

sns.lineplot(
    data=plot_long,
    x="date",
    y="event_count",
    hue="variant_key",
    ax=axes[1],
    linewidth=1.5,
)
axes[1].set_title("Daily event_count by variant")
axes[1].set_xlabel("date")
axes[1].set_ylabel("event_count")

plt.tight_layout()
plt.show()

In [ ]:
keep_long_cols = [
    "date",
    "stage2_split",
    "is_main_stage2_sample",
    "variant_key",
    "gold_close",
    "gold_ret_1d",
    "max_p_taco",
    "sum_p_taco",
    "mean_p_taco",
    "event_count",
    "high_p_taco_count",
    "p_taco_top1",
    "p_taco_top2_sum",
    "p_taco_any",
    "p_taco_tail_005",
    "p_taco_tail_010",
    "event_count_ge_005",
    "event_count_ge_010",
    "event_count_ge_020",
    "p_taco_any_decay_2d",
    "p_taco_any_decay_3d",
    "p_taco_top1_decay_2d",
    "p_taco_top1_decay_3d",
    "p_taco_top2_sum_decay_2d",
    "p_taco_top2_sum_decay_3d",
    "p_taco_tail_005_decay_2d",
    "p_taco_tail_005_decay_3d",
    "p_taco_tail_010_decay_2d",
    "p_taco_tail_010_decay_3d",
    "p_taco_any_x_vix_ma_5d_lag1",
    "p_taco_any_x_spx_drawdown_5d_lag1",
    "p_taco_any_x_dxy_ret_1d_lag1",
    "has_stage1_signal",
    "holdout_spillover_flag",
    "dxy_ret_1d_lag1",
    "real_yield_change_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
    "trend_tariff_z_60d_lag1",
    "trend_inflation_z_60d_lag1",
    "trend_war_z_60d_lag1",
    "trend_tariff_spike_lag1",
    "trend_inflation_spike_lag1",
    "trend_war_spike_lag1",
    "source_positive_count",
    "prediction_source_first",
    "min_event_time_utc",
    "max_event_time_utc",
]
stage2_long_export = stage2_long[keep_long_cols].copy()

stage1_events.to_csv(BRIDGE_PATH, index=False)
stage2_long_export.to_csv(LONG_PATH, index=False)
stage2_wide.to_csv(WIDE_PATH, index=False)

print("Saved:")
print(" -", BRIDGE_PATH)
print(" -", LONG_PATH)
print(" -", WIDE_PATH)

## What Comes Next

这版特征工程完成后，后续 `Stage 2` notebook 可以直接基于：
- `stage2_daily_features_long_v2.csv`
- `stage2_daily_features_wide_v2.csv`

进行：
1. `Random Walk / Macro-only / Single-stage / Two-stage` 的建模对照
2. 运行时指定的 `P_taco` variant 建模比较
3. `legacy v1 aggregation` 与 `aggregation v2` 的对照
4. `raw vs calibrated`、`with vs without FinBERT` 等后续消融